<a href="https://colab.research.google.com/github/arman-taghizadeh/zero-shot-diffusion-object-removal/blob/main/object_removal_demo.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#### License

This implementation is released under the
[PolyForm Noncommercial License 1.0.0](LICENSE).

Use, modification, and distribution are permitted only for
noncommercial purposes as defined by the license. Commercial use
requires separate permission from the copyright holder.

The pretrained models and third-party libraries used by this notebook
remain subject to their respective licenses and terms of use.

In [ ]:
# Copyright © 2026 Arman Taghizadeh
# Licensed under the PolyForm Noncommercial License 1.0.0.

#### Setup

This notebook is designed for Google Colab with a GPU runtime.

On the first run, execute the dependency cell below and restart the
Colab session if prompted. Then select **Runtime → Run all**.

The implementation was tested with:
Python 3.13.15
PyTorch 2.11.0+cu128
torchvision 0.26.0+cu128
numpy 2.1.3
Pillow 11.3.0
OpenCV 4.14.0
matplotlib 3.11.1
transformers 5.15.1
diffusers 0.40.0
accelerate 1.14.0
safetensors 0.8.0

In [ ]:
# ============================================================
# Setup and Dependencies
# ============================================================

%pip install -q \
    "numpy==2.1.3" \
    "Pillow==11.3.0" \
    "opencv-python-headless==4.14.0.94" \
    "matplotlib==3.11.1" \
    "tqdm==4.67.3" \
    "diffusers==0.40.0" \
    "transformers==5.15.1" \
    "accelerate==1.14.0" \
    "safetensors==0.8.0" \
    "lpips==0.1.4"

%pip install -q \
    "git+https://github.com/facebookresearch/segment-anything.git@dca509fe793f601edb92606367a655c15ac00fdf"

In [ ]:
import os, io, base64
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
from IPython.display import display, HTML
from google.colab import output as colab_output

# ----------------------------
# 1) Download SAM checkpoint (once)
# ----------------------------
SAM_CKPT = "sam_vit_h_4b8939.pth"
if not os.path.exists(SAM_CKPT):
    !wget -q https://dl.fbaipublicfiles.com/segment_anything/sam_vit_h_4b8939.pth -O sam_vit_h_4b8939.pth
print("SAM checkpoint:", SAM_CKPT)

In [ ]:
# ============================================================
# Configuration
# ============================================================

# Reproducibility
SEED = 0

# Image / model
IMAGE_SIZE = 512
MODEL_ID = "CompVis/stable-diffusion-v1-4"
BLIP_ID = "Salesforce/blip-image-captioning-base"

# Diffusion
NUM_STEPS = 50

# Mask postprocessing
MASK_DILATION_KERNEL = 9

# Masked null-text optimization
CFG_W_TRAIN = 7.5
NTI_ITERS = 5
NTI_LR = 1e-2
LAMBDA_IN = 0.001
EPS_STOP = 1e-5

# Decoder self-attention masking
SAM_STEPS = 9
SAM_SCALE = 0.3
HAM_PEAK = 1.0
HAM_START_FRAC = 0.25
HAM_RAMP_END_FRAC = 0.80

# Primary editing
CFG_W = 9.5

# Refinement
REFINE_ROUNDS = 10
TREF_FRAC = 0.5
CFG_REF = 9.5
SEED_REFINE = 0

# Evaluation
BAND_PX = 12
RING_PX = 20
RING_SEAM_PX = 8

In [ ]:
import os, gc, math, random
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt

from tqdm.auto import tqdm
from diffusers import StableDiffusionPipeline, DDIMScheduler
#from transformers import pipeline as hf_pipeline

device = "cuda" if torch.cuda.is_available() else "cpu"
torch.manual_seed(SEED)
np.random.seed(SEED)
random.seed(SEED)
print("device:", device)


In [ ]:
import sys
import cv2
import PIL
import torch
import torchvision
import matplotlib
import transformers
import diffusers
import accelerate
import safetensors

print("Python:", sys.version.split()[0])
print("torch:", torch.__version__)
print("torchvision:", torchvision.__version__)
print("numpy:", np.__version__)
print("Pillow:", PIL.__version__)
print("OpenCV:", cv2.__version__)
print("matplotlib:", matplotlib.__version__)
print("transformers:", transformers.__version__)
print("diffusers:", diffusers.__version__)
print("accelerate:", accelerate.__version__)
print("safetensors:", safetensors.__version__)

In [ ]:
# ============================================================
# Input Image
# ============================================================

from google.colab import files

uploaded = files.upload()
img_path = next(iter(uploaded.keys()))
print("Image:", img_path)

img_pil = Image.open(img_path).convert("RGB")
img_pil_512 = img_pil.resize(
    (IMAGE_SIZE, IMAGE_SIZE),
    Image.BILINEAR
)

plt.figure(figsize=(6,6))
plt.imshow(img_pil_512)
plt.axis("off")
plt.title("Input (512x512)")
plt.show()


In [ ]:
# ============================================================
# Interactive Object Mask Construction (SAM + Brush Mask Editor)

#   - Step 1: Click points -> SAM mask
#   - Step 2: Brush-edit OBJECT mask (add/erase) with TRANSPARENT brush overlay
# ============================================================


# ----------------------------
# Load SAM model + predictor
# ----------------------------
from segment_anything import sam_model_registry, SamPredictor

sam = sam_model_registry["vit_h"](checkpoint=SAM_CKPT).to(device)
predictor = SamPredictor(sam)

np_img = np.array(img_pil_512)  # HWC RGB, 512x512
predictor.set_image(np_img)

# ----------------------------
# Helper: PIL -> b64 png
# ----------------------------
def pil_to_b64_png(pil_img: Image.Image) -> str:
    buf = io.BytesIO()
    pil_img.save(buf, format="PNG")
    return base64.b64encode(buf.getvalue()).decode("utf-8")

img_b64 = pil_to_b64_png(img_pil_512)

# ============================================================
# Step 1 — Collect points
# ============================================================
display(HTML("""
<script>
window._sam_points = [];
window._sam_done = false;
window._sam_resolve = null;
</script>
"""))

points_html = f"""
<div style="display:flex; gap:16px; align-items:flex-start; flex-wrap:wrap;">
  <div>
    <div style="font-weight:600; margin-bottom:8px;">
      Step 1 — Click <b>positive</b> points inside the object. Undo/Clear if needed, then press <b>Done</b>.
    </div>
    <div style="position:relative; display:inline-block; border:1px solid #ddd;">
      <img id="sam_img" src="data:image/png;base64,{img_b64}" style="display:block; width:512px; height:512px;"/>
      <canvas id="sam_canvas" style="position:absolute; left:0; top:0; width:512px; height:512px;"></canvas>
    </div>
    <div style="margin-top:10px; display:flex; gap:8px;">
      <button id="sam_done"  style="padding:6px 10px;">Done</button>
      <button id="sam_undo"  style="padding:6px 10px;">Undo</button>
      <button id="sam_clear" style="padding:6px 10px;">Clear</button>
    </div>
    <div style="color:#666; font-size:12px; margin-top:8px;">
      Tip: click multiple points inside the object.
    </div>
  </div>

  <div style="min-width:240px;">
    <div style="font-weight:600; margin-bottom:8px;">Points (x, y):</div>
    <pre id="sam_points" style="background:#f7f7f7; padding:10px; border:1px solid #ddd; height:260px; overflow:auto;"></pre>
  </div>
</div>

<script>
(() => {{
  const canvas = document.getElementById("sam_canvas");
  const ctx = canvas.getContext("2d");
  const pre = document.getElementById("sam_points");
  const btnDone = document.getElementById("sam_done");
  const btnUndo = document.getElementById("sam_undo");
  const btnClear = document.getElementById("sam_clear");

  canvas.width = 512;
  canvas.height = 512;

  function redraw() {{
    ctx.clearRect(0, 0, 512, 512);
    ctx.fillStyle = "rgba(255,0,0,0.9)";
    ctx.strokeStyle = "rgba(255,255,255,0.9)";
    ctx.lineWidth = 2;

    for (const p of window._sam_points) {{
      ctx.beginPath();
      ctx.arc(p.x, p.y, 5, 0, Math.PI * 2);
      ctx.fill();
      ctx.stroke();
    }}
    pre.textContent = window._sam_points.map(p => `(${{p.x}}, ${{p.y}})`).join("\\n");
  }}

  function getXY(e) {{
    const r = canvas.getBoundingClientRect();
    const x = Math.round((e.clientX - r.left) * (512 / r.width));
    const y = Math.round((e.clientY - r.top)  * (512 / r.height));
    return {{x, y}};
  }}

  canvas.addEventListener("click", (e) => {{
    const {{x, y}} = getXY(e);
    window._sam_points.push({{x, y}});
    redraw();
  }});

  btnUndo.addEventListener("click", () => {{
    if (window._sam_points.length) {{
      window._sam_points.pop();
      redraw();
    }}
  }});

  btnClear.addEventListener("click", () => {{
    window._sam_points = [];
    redraw();
  }});

  btnDone.addEventListener("click", () => {{
    window._sam_done = true;
    if (window._sam_resolve) {{
      window._sam_resolve(window._sam_points.map(p => [p.x, p.y]));
      window._sam_resolve = null;
    }}
  }});

  redraw();
}})();
</script>
"""
display(HTML(points_html))
print("➡️ Click points then press Done (cell continues automatically).")

pts = colab_output.eval_js("""
new Promise(resolve => {
  if (window._sam_done) resolve(window._sam_points.map(p => [p.x, p.y]));
  else window._sam_resolve = resolve;
})
""")

points = np.array(pts, dtype=np.int32)
assert points.shape[0] >= 1, "Need at least 1 point."
labels = np.ones((points.shape[0],), dtype=np.int32)
print("Collected points:", points.tolist())

# ----------------------------
# Run SAM prediction (choose best mask by score)
# ----------------------------
masks, scores, logits = predictor.predict(
    point_coords=points,
    point_labels=labels,
    multimask_output=True,
)
best = int(np.argmax(scores))
mask_obj_sam = masks[best].astype(np.uint8)  # 0/1
print("SAM scores:", scores.tolist(), "| chosen:", best, "score=", float(scores[best]))

# ============================================================
# Shared helper: Brush UI builder (returns a PNG dataURL)
# - Transparent brush preview (not solid)
# - Erase mode: Shift-drag OR right-drag OR toggle button
# ============================================================
_BRUSH_UI_COUNTER = 0

def run_brush_ui(mask_init_b64: str, title: str, overlay_rgba=(255,0,0,120)):
    global _BRUSH_UI_COUNTER
    _BRUSH_UI_COUNTER += 1
    uid = f"brush_{_BRUSH_UI_COUNTER}"

    r, g, b, a = overlay_rgba

    display(HTML(f"""
    <script>
    window._mask_done_{uid} = false;
    window._mask_resolve_{uid} = null;
    </script>
    """))

    brush_html = f"""
    <div style="display:flex; gap:16px; align-items:flex-start; flex-wrap:wrap; margin-top:18px;">
      <div>
        <div style="font-weight:600; margin-bottom:8px;">{title}</div>

        <div style="position:relative; display:inline-block; border:1px solid #ddd; user-select:none;">
          <img src="data:image/png;base64,{img_b64}"
               style="display:block; width:512px; height:512px; pointer-events:none;"/>

          <canvas id="{uid}_overlay"
                  style="position:absolute; left:0; top:0; width:512px; height:512px; touch-action:none;"></canvas>

          <canvas id="{uid}_mask_canvas" style="display:none;"></canvas>
        </div>

        <div style="margin-top:10px; display:flex; gap:10px; align-items:center; flex-wrap:wrap;">
          <label style="display:flex; gap:8px; align-items:center;">
            Brush size:
            <input id="{uid}_brush_size" type="range" min="2" max="80" value="22"/>
            <span id="{uid}_brush_val">22</span>
          </label>

          <button id="{uid}_mode_btn" style="padding:6px 10px; font-weight:600;">
            Mode: ADD
          </button>

          <span style="color:#666; font-size:12px;">
            Shortcuts: Erase = hold <b>Shift</b> or <b>right-drag</b>
          </span>
        </div>

        <div style="margin-top:10px; display:flex; gap:8px; flex-wrap:wrap;">
          <button id="{uid}_btn_reset" style="padding:6px 10px;">Reset to SAM</button>
          <button id="{uid}_btn_clear" style="padding:6px 10px;">Clear mask</button>
          <button id="{uid}_btn_done"  style="padding:6px 10px; font-weight:600;">Done</button>
        </div>
      </div>
    </div>

    <script>
    (() => {{
      const overlay = document.getElementById("{uid}_overlay");
      const octx = overlay.getContext("2d");

      const maskCanvas = document.getElementById("{uid}_mask_canvas");
      const mctx = maskCanvas.getContext("2d");

      overlay.width = 512; overlay.height = 512;
      maskCanvas.width = 512; maskCanvas.height = 512;

      const slider = document.getElementById("{uid}_brush_size");
      const bval   = document.getElementById("{uid}_brush_val");
      const btnReset = document.getElementById("{uid}_btn_reset");
      const btnClear = document.getElementById("{uid}_btn_clear");
      const btnDone  = document.getElementById("{uid}_btn_done");
      const modeBtn  = document.getElementById("{uid}_mode_btn");

      let brush = parseInt(slider.value);
      bval.textContent = String(brush);

      // persistent mode toggle
      let forceEraseMode = false;
      function updateModeBtn() {{
        modeBtn.textContent = forceEraseMode ? "Mode: ERASE" : "Mode: ADD";
        modeBtn.style.background = forceEraseMode ? "#ffe6e6" : "#e9f7ff";
        modeBtn.style.border = "1px solid #ccc";
      }}
      updateModeBtn();

      modeBtn.addEventListener("click", () => {{
        forceEraseMode = !forceEraseMode;
        updateModeBtn();
      }});

      const initMaskImg = new Image();
      initMaskImg.onload = () => {{
        mctx.clearRect(0,0,512,512);
        mctx.drawImage(initMaskImg, 0, 0, 512, 512);
        redrawOverlayFromMask();
      }};
      initMaskImg.src = "data:image/png;base64,{mask_init_b64}";

      function redrawOverlayFromMask() {{
        const imgData = mctx.getImageData(0,0,512,512);
        const d = imgData.data;

        const out = octx.createImageData(512,512);
        const od = out.data;

        for (let i=0; i<d.length; i+=4) {{
          const v = d[i];
          if (v > 127) {{
            od[i+0] = {r};
            od[i+1] = {g};
            od[i+2] = {b};
            od[i+3] = {a};   // semi-transparent mask overlay
          }} else {{
            od[i+3] = 0;
          }}
        }}

        octx.clearRect(0,0,512,512);
        octx.putImageData(out, 0, 0);
      }}

      function getXY(e) {{
        const rr = overlay.getBoundingClientRect();
        const x = Math.round((e.clientX - rr.left) * (512 / rr.width));
        const y = Math.round((e.clientY - rr.top)  * (512 / rr.height));
        return {{x: Math.max(0, Math.min(511, x)), y: Math.max(0, Math.min(511, y))}};
      }}

      function isEraseEvent(e) {{
        // ✅ erase if: toggled OR shift OR right button drag
        return forceEraseMode || e.shiftKey || (e.button === 2) || (e.buttons === 2);
      }}

      function paintMaskAt(x, y, isErase) {{
        mctx.save();
        mctx.globalCompositeOperation = "source-over";
        mctx.fillStyle = isErase ? "rgb(0,0,0)" : "rgb(255,255,255)";
        mctx.beginPath();
        mctx.arc(x, y, brush, 0, Math.PI*2);
        mctx.fill();
        mctx.restore();
      }}

      // ✅ Transparent cursor preview (does NOT accumulate into solid)
      function drawCursorPreview(x, y, isErase) {{
        redrawOverlayFromMask();

        octx.save();
        octx.globalCompositeOperation = "source-over";

        if (!isErase) {{
          // translucent brush fill
          octx.fillStyle = "rgba({r},{g},{b},0.22)";
          octx.strokeStyle = "rgba(255,255,255,0.9)";
          octx.lineWidth = 2;
          octx.beginPath();
          octx.arc(x, y, brush, 0, Math.PI*2);
          octx.fill();
          octx.stroke();
        }} else {{
          // eraser preview: ring only
          octx.strokeStyle = "rgba(255,255,255,0.95)";
          octx.lineWidth = 2;
          octx.beginPath();
          octx.arc(x, y, brush, 0, Math.PI*2);
          octx.stroke();
        }}

        octx.restore();
      }}

      let drawing = false;
      let last = null;

      function drawLine(a, b, isErase) {{
        const dx = b.x - a.x;
        const dy = b.y - a.y;
        const dist = Math.sqrt(dx*dx + dy*dy);
        const step = Math.max(1, Math.floor(dist / (brush * 0.35)));
        for (let i=0; i<=step; i++) {{
          const t = i / step;
          const x = Math.round(a.x + t*dx);
          const y = Math.round(a.y + t*dy);
          paintMaskAt(x, y, isErase);
        }}
      }}

      overlay.addEventListener("contextmenu", (e) => e.preventDefault());

      overlay.addEventListener("pointerdown", (e) => {{
        drawing = true;
        overlay.setPointerCapture(e.pointerId);
        const p = getXY(e);
        const isErase = isEraseEvent(e);
        paintMaskAt(p.x, p.y, isErase);
        drawCursorPreview(p.x, p.y, isErase);
        last = p;
      }});

      overlay.addEventListener("pointermove", (e) => {{
        const p = getXY(e);
        const isErase = isEraseEvent(e);

        if (drawing) {{
          if (last) drawLine(last, p, isErase);
          last = p;
        }}

        drawCursorPreview(p.x, p.y, isErase);
      }});

      overlay.addEventListener("pointerup", () => {{
        drawing = false;
        last = null;
        redrawOverlayFromMask();
      }});

      overlay.addEventListener("pointerleave", () => {{
        if (!drawing) redrawOverlayFromMask();
      }});

      slider.addEventListener("input", () => {{
        brush = parseInt(slider.value);
        bval.textContent = String(brush);
        redrawOverlayFromMask();
      }});

      btnReset.addEventListener("click", () => {{
        mctx.clearRect(0,0,512,512);
        mctx.drawImage(initMaskImg, 0, 0, 512, 512);
        redrawOverlayFromMask();
      }});

      btnClear.addEventListener("click", () => {{
        mctx.clearRect(0,0,512,512);
        redrawOverlayFromMask();
      }});

      btnDone.addEventListener("click", () => {{
        window._mask_done_{uid} = true;
        const dataURL = maskCanvas.toDataURL("image/png");
        if (window._mask_resolve_{uid}) {{
          window._mask_resolve_{uid}(dataURL);
          window._mask_resolve_{uid} = null;
        }}
      }});
    }})();
    </script>
    """

    display(HTML(brush_html))
    print("➡️ Brush (transparent preview). ADD by drag. ERASE by Shift-drag / right-drag / Mode button. Then Done.")

    dataurl = colab_output.eval_js(f"""
    new Promise(resolve => {{
      if (window._mask_done_{uid}) resolve(null);
      else window._mask_resolve_{uid} = resolve;
    }})
    """)

    assert isinstance(dataurl, str) and dataurl.startswith("data:image/png;base64,"), "Mask export failed."
    return dataurl

def dataurl_to_mask01(dataurl: str) -> np.ndarray:
    b64 = dataurl.split(",", 1)[1]
    mask_bytes = base64.b64decode(b64)
    pil = Image.open(io.BytesIO(mask_bytes)).convert("L")
    arr = np.array(pil)
    return (arr > 127).astype(np.uint8)

# ============================================================
# Step 2 — OBJECT brush mask (red)
# ============================================================
mask_init_obj_pil = Image.fromarray((mask_obj_sam * 255).astype(np.uint8), mode="L")
mask_init_obj_b64 = pil_to_b64_png(mask_init_obj_pil)

obj_dataurl = run_brush_ui(
    mask_init_b64=mask_init_obj_b64,
    title="Step 2 — Brush-edit OBJECT mask (RED, transparent overlay). Cover the object body.",
    overlay_rgba=(255, 0, 0, 120),
)
mask_obj = dataurl_to_mask01(obj_dataurl)

print("OBJECT mask stats:",
      "area=", float(mask_obj.mean()),
      "unique=", np.unique(mask_obj).tolist())

# Keep compatibility with later cells:
mask_dil = mask_obj

# ----------------------------
# Visualize overlays
# ----------------------------
plt.figure(figsize=(12,5))

plt.subplot(1,2,1)
plt.imshow(np_img); plt.imshow(mask_obj_sam, alpha=0.5)
plt.axis("off"); plt.title("SAM mask (raw)")

plt.subplot(1,2,2)
plt.imshow(np_img); plt.imshow(mask_obj, alpha=0.5)
plt.axis("off"); plt.title("OBJECT mask (after brush)")

plt.show()


In [ ]:

# ============================================================
# Mask Postprocessing + Latent/Token Masks
# ============================================================


assert "mask_obj" in globals(), "mask_obj missing from Cell 3"
np_img = np.array(img_pil_512)

# ----------------------------
# Helpers
# ----------------------------
def keep_largest_cc(binary_mask: np.ndarray) -> np.ndarray:
    num, labels, stats, _ = cv2.connectedComponentsWithStats(binary_mask.astype(np.uint8), connectivity=8)
    if num <= 1:
        return binary_mask.astype(np.uint8)
    areas = stats[1:, cv2.CC_STAT_AREA]  # skip background
    best = 1 + int(np.argmax(areas))
    return (labels == best).astype(np.uint8)

def fill_holes(binary_mask: np.ndarray) -> np.ndarray:
    m = (binary_mask.astype(np.uint8) * 255)
    h, w = m.shape[:2]
    flood = m.copy()
    ff_mask = np.zeros((h + 2, w + 2), np.uint8)
    cv2.floodFill(flood, ff_mask, seedPoint=(0, 0), newVal=255)
    flood_inv = cv2.bitwise_not(flood)
    filled = cv2.bitwise_or(m, flood_inv)
    return (filled > 0).astype(np.uint8)

# ----------------------------
# 1) Clean + postprocess
# ----------------------------
mask_bin = (mask_obj > 0).astype(np.uint8)

mask_cc     = keep_largest_cc(mask_bin)
mask_filled = fill_holes(mask_cc)

# Dilate a bit
kernel = np.ones(
    (MASK_DILATION_KERNEL, MASK_DILATION_KERNEL),
    np.uint8
)
mask_dil = cv2.dilate(mask_filled, kernel, iterations=1).astype(np.uint8)  # FINAL hard mask

# ----------------------------
# 2) Torch masks
# ----------------------------
mask_img_hard = torch.from_numpy(mask_dil[None, None]).float()  # [1,1,512,512]

# latent masks (64x64)
mask_latent_hard = F.interpolate(mask_img_hard, size=(64, 64), mode="nearest")

# downstream names used by later cells
mask_in_hard  = mask_latent_hard.to(device=device, dtype=torch.float32).clamp(0, 1)
mask_out_hard = (1.0 - mask_in_hard).clamp(0, 1)

# token masks for HAM/SAM
def token_mask_from_latent(mask_latent_hard, res):
    m = F.interpolate(mask_latent_hard, size=(res, res), mode="nearest")  # [1,1,res,res]
    return m.flatten(2).squeeze(1)  # [1, res*res]

mask_tok_8  = token_mask_from_latent(mask_latent_hard, 8)
mask_tok_16 = token_mask_from_latent(mask_latent_hard, 16)
mask_tok_32 = token_mask_from_latent(mask_latent_hard, 32)
mask_tok_64 = token_mask_from_latent(mask_latent_hard, 64)

# ----------------------------
# Visualize (debug)
# ----------------------------
plt.figure(figsize=(14, 5))

plt.subplot(1, 3, 1)
plt.imshow(np_img)
plt.imshow(mask_cc, alpha=0.5)
plt.axis("off")
plt.title("Largest CC")

plt.subplot(1, 3, 2)
plt.imshow(np_img)
plt.imshow(mask_filled, alpha=0.5)
plt.axis("off")
plt.title("After hole fill")

plt.subplot(1, 3, 3)
plt.imshow(np_img)
plt.imshow(mask_dil, alpha=0.5)
plt.axis("off")
plt.title("Filled + dilated (FINAL hard mask)")

plt.tight_layout()
plt.show()

print("mask area (img hard mean):", float(mask_img_hard.mean()))


In [ ]:
# ============================================================
# Automatic Image Captioning

#   - Generate caption automatically using BLIP
# ============================================================

from transformers import BlipProcessor, BlipForConditionalGeneration



processor = BlipProcessor.from_pretrained(BLIP_ID)
blip = BlipForConditionalGeneration.from_pretrained(BLIP_ID).to(device)
blip.eval()

# ---- caption generation ----
with torch.no_grad():
    inputs = processor(images=img_pil_512, return_tensors="pt").to(device)
    out_ids = blip.generate(
        **inputs,
        max_new_tokens=30,
        num_beams=5,
    )

caption = processor.decode(out_ids[0], skip_special_tokens=True).strip()
print("Blip Caption:", caption)


p_base = caption
# p_pos  = caption

print("p_base:", p_base)
# print("p_pos :", p_pos)


In [ ]:
# ============================================================
# Stable Diffusion Initialization
# ============================================================

import contextlib

assert "img_pil_512" in globals(), "img_pil_512 missing"
assert "p_base" in globals(), "p_base missing"
assert "mask_in_hard" in globals() and "mask_out_hard" in globals(), "Run Cell 4 first (mask_in/out_hard missing)"
assert all(k in globals() for k in ["mask_tok_8","mask_tok_16","mask_tok_32","mask_tok_64"]), "Run Cell 4 first (token masks missing)"

# ----------------------------
# 1) Load SD1.4
# ----------------------------
# MODEL_ID = "CompVis/stable-diffusion-v1-4"

pipe = StableDiffusionPipeline.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.float16 if device == "cuda" else torch.float32,
    safety_checker=None,
    requires_safety_checker=False,
).to(device)

pipe.scheduler = DDIMScheduler.from_config(pipe.scheduler.config)

tokenizer    = pipe.tokenizer
text_encoder = pipe.text_encoder
unet         = pipe.unet
vae          = pipe.vae

DTYPE = unet.dtype

# VAE tiling/slicing off (seams/stripes)
try: pipe.vae.disable_slicing()
except Exception: pass
try: pipe.vae.disable_tiling()
except Exception: pass

# ----------------------------
# 2) Debug helpers
# ----------------------------
def _tinfo(name, x):
    x = x.detach()
    finite = bool(torch.isfinite(x).all())
    x2 = torch.nan_to_num(x.float(), nan=0.0, posinf=0.0, neginf=0.0)
    print(f"{name:18s} shape={tuple(x.shape)} dtype={x.dtype} "
          f"finite={finite} min={float(x2.min()):.4f} max={float(x2.max()):.4f} mean={float(x2.mean()):.4f}")

def _imginfo(name, img_01_bchw):
    x = img_01_bchw.detach()
    finite = bool(torch.isfinite(x).all())
    x2 = torch.nan_to_num(x.float(), nan=0.0, posinf=1.0, neginf=0.0)
    print(f"{name:18s} shape={tuple(x.shape)} dtype={x.dtype} finite={finite} "
          f"min={float(x2.min()):.4f} max={float(x2.max()):.4f} mean={float(x2.mean()):.4f}")

def _to_pil(x_chw):
    x = x_chw.permute(1,2,0).numpy()
    x = np.nan_to_num(x, nan=0.0, posinf=1.0, neginf=0.0)
    return Image.fromarray((x*255).clip(0,255).astype(np.uint8))

# ----------------------------
# 3) Text embeddings
# ----------------------------
def encode_prompt(prompt: str):
    tok = tokenizer(
        prompt,
        padding="max_length",
        max_length=tokenizer.model_max_length,
        truncation=True,
        return_tensors="pt",
    )
    with torch.no_grad():
        emb = text_encoder(tok.input_ids.to(device))[0]
    return emb.to(dtype=DTYPE)

emb_cond        = encode_prompt(p_base)
emb_uncond_init = encode_prompt("")

# ----------------------------
# 4) FP32-safe VAE encode/decode
# ----------------------------
def _vae_fp32_ctx():
    if device == "cuda":
        return torch.autocast("cuda", enabled=False)
    return contextlib.nullcontext()

@torch.no_grad()
def vae_encode_01_fp32(img_pil_512):
    arr = np.array(img_pil_512).astype(np.float32) / 255.0
    x = torch.from_numpy(arr).permute(2,0,1)[None].to(device=device, dtype=torch.float32)
    x = x * 2 - 1

    old_dtype = pipe.vae.dtype
    pipe.vae.to(dtype=torch.float32)
    with _vae_fp32_ctx():
        lat = pipe.vae.encode(x).latent_dist.sample() * 0.18215
    pipe.vae.to(dtype=old_dtype)
    return lat.to(dtype=DTYPE)

@torch.no_grad()
def vae_decode_fp32(lat):
    old_dtype = pipe.vae.dtype
    pipe.vae.to(dtype=torch.float32)
    with _vae_fp32_ctx():
        lat32 = lat.to(device=device, dtype=torch.float32)
        x = pipe.vae.decode(lat32 / 0.18215).sample
        x = (x/2 + 0.5).clamp(0,1)
    pipe.vae.to(dtype=old_dtype)
    return x

@torch.no_grad()
def vae_decode(lat):
    x = vae.decode(lat / 0.18215).sample
    x = (x/2 + 0.5).clamp(0,1)
    return x

x0_latent = vae_encode_01_fp32(img_pil_512)

# ----------------------------
# 5) Masks -> device/dtype (ONLY hard masks)
# ----------------------------
mask_in_hard  = mask_in_hard.to(device=device, dtype=DTYPE).clamp(0,1)
mask_out_hard = mask_out_hard.to(device=device, dtype=DTYPE).clamp(0,1)

mask_tok_8  = mask_tok_8.to(device=device, dtype=DTYPE)
mask_tok_16 = mask_tok_16.to(device=device, dtype=DTYPE)
mask_tok_32 = mask_tok_32.to(device=device, dtype=DTYPE)
mask_tok_64 = mask_tok_64.to(device=device, dtype=DTYPE)

# ----------------------------
# 6) Sanity
# ----------------------------
print("=== Cell 6 Sanity ===")
_tinfo("x0_latent", x0_latent)
_tinfo("emb_cond", emb_cond)
_tinfo("emb_uncond_init", emb_uncond_init)
_tinfo("mask_in_hard", mask_in_hard)
_tinfo("mask_out_hard", mask_out_hard)

with torch.no_grad():
    mh = mask_in_hard[0,0].float().cpu().numpy()
plt.figure(figsize=(5,5))
plt.imshow(mh); plt.title("mask_in_hard (64x64)"); plt.axis("off")
plt.show()

# stripe diagnostic (optional)
with torch.no_grad():
    img16 = vae_decode(x0_latent)[0].float().cpu()
    img32 = vae_decode_fp32(x0_latent)[0].float().cpu()
_imginfo("decode_fp16(x0)", img16[None])
_imginfo("decode_fp32(x0)", img32[None])

plt.figure(figsize=(12,6))
plt.subplot(1,2,1); plt.imshow(_to_pil(img16)); plt.axis("off"); plt.title("x0 decode fp16 (diagnostic)")
plt.subplot(1,2,2); plt.imshow(_to_pil(img32)); plt.axis("off"); plt.title("x0 decode fp32 (diagnostic)")
plt.tight_layout()
plt.show()


In [ ]:
# ============================================================
# DDIM Inversion
# ============================================================

from tqdm import tqdm


@torch.no_grad()
def ddim_invert_true_aligned(x0_latent, emb_cond, num_steps=NUM_STEPS):
    sch = pipe.scheduler
    sch.set_timesteps(num_steps, device=device)
    ts = sch.timesteps  # descending

    inv_xt = [None] * len(ts)
    inv_xt[-1] = x0_latent.clone()

    for i in tqdm(range(len(ts)-1, 0, -1), desc="DDIM inversion TRUE (clean->noisy)"):
        t      = ts[i]
        t_prev = ts[i-1]

        x_t = inv_xt[i]

        eps = unet(x_t.to(DTYPE), t, encoder_hidden_states=emb_cond).sample
        a_t = sch.alphas_cumprod[t]
        a_p = sch.alphas_cumprod[t_prev]

        x0_hat = (x_t - torch.sqrt(1 - a_t) * eps) / torch.sqrt(a_t)
        x_prev = torch.sqrt(a_p) * x0_hat + torch.sqrt(1 - a_p) * eps
        inv_xt[i-1] = x_prev.clone()

    return inv_xt, ts

# NUM_STEPS = 50
inv_xt, timesteps = ddim_invert_true_aligned(x0_latent, emb_cond, num_steps=NUM_STEPS)
# xT_inv = inv_xt[0].clone()

print("=== Cell 7 Sanity (TRUE inversion) ===")
print("timesteps:", len(timesteps), "| inv_xt:", len(inv_xt))
_tinfo("inv_xt[0] (xT/noisy)", inv_xt[0])
_tinfo("inv_xt[-1] (x0/clean)", inv_xt[-1])

with torch.no_grad():
    mse_x0 = torch.mean((inv_xt[-1] - x0_latent)**2).item()
print(f"[CHK] MSE(inv_xt[-1], x0_latent) = {mse_x0:.6e}  (should be ~0)")

@torch.no_grad()
def inversion_forward_consistency(inv_xt, ts, emb_cond, ncheck=6):
    sch = pipe.scheduler
    idxs = np.linspace(1, len(ts)-1, ncheck).astype(int).tolist()
    print("Forward-consistency idxs (i -> i-1):", idxs)
    for i in idxs:
        t      = ts[i]
        t_prev = ts[i-1]
        x_t = inv_xt[i]

        eps = unet(x_t.to(DTYPE), t, encoder_hidden_states=emb_cond).sample
        a_t = sch.alphas_cumprod[t]
        a_p = sch.alphas_cumprod[t_prev]
        x0_hat = (x_t - torch.sqrt(1 - a_t) * eps) / torch.sqrt(a_t)
        x_prev_hat = torch.sqrt(a_p) * x0_hat + torch.sqrt(1 - a_p) * eps

        mse = torch.mean((x_prev_hat - inv_xt[i-1])**2).item()
        print(f"  i={i:02d} (t={int(t):4d})->(t_prev={int(t_prev):4d}) MSE={mse:.6e}")

inversion_forward_consistency(inv_xt, timesteps, emb_cond, ncheck=6)

with torch.no_grad():
    img_clean = vae_decode_fp32(inv_xt[-1])[0].float().cpu()
    img_noisy = vae_decode_fp32(inv_xt[0])[0].float().cpu()

plt.figure(figsize=(12,6))
plt.subplot(1,2,1); plt.imshow(_to_pil(img_clean)); plt.axis("off"); plt.title("inv_xt[-1] (should match input, clean)")
plt.subplot(1,2,2); plt.imshow(_to_pil(img_noisy)); plt.axis("off"); plt.title("inv_xt[0] (noisy)")
plt.show()


In [ ]:
# ============================================================
# Background-Weighted Masked Null-Text Optimization
# ============================================================

# ----------------------------
# Core knobs (TRAIN nulls)
# ----------------------------
# CFG_W_TRAIN = 7.5

# NTI_ITERS = 5
# NTI_LR    = 1e-2
NTI_CLAMP = None

LOSS_MODE = "outside+tiny-inside"   # or "full"
# LAMBDA_IN = 0.001                   # tiny reg on object-inside only (hard)
# EPS_STOP  = 1e-5

LAMBDA_U_SMOOTH = 0.0
USE_FP32_UNET_OPT = True

SKIP_IF_GOOD = False
SKIP_LOSS_OUT_THR_BASE = 1e-4
SKIP_THR_LATE_MULT     = 10.0
LATE_START_FRAC        = 0.35

# ----------------------------
# Setup
# ----------------------------
pipe.scheduler.set_timesteps(NUM_STEPS, device=device)
ts = pipe.scheduler.timesteps
assert len(inv_xt) == len(ts) == NUM_STEPS, "inv_xt / ts length mismatch."

# ✅ Masks (HARD ONLY)
# mask_in_hard   = object region (0/1)
# mask_out_hard  = outside object (0/1)
mask_in_hard  = mask_in_hard.to(device=device, dtype=DTYPE).clamp(0,1)
mask_out_hard = mask_out_hard.to(device=device, dtype=DTYPE).clamp(0,1)

def _skip_thr_for_k(k, N):
    if k >= int(LATE_START_FRAC * (N - 1)):
        return float(SKIP_LOSS_OUT_THR_BASE) * float(SKIP_THR_LATE_MULT)
    return float(SKIP_LOSS_OUT_THR_BASE)

def is_finite(x):
    return bool(torch.isfinite(x).all())

_unet_dtype_orig = unet.dtype

def _set_unet_dtype(dtype: torch.dtype):
    global unet
    if unet.dtype != dtype:
        unet.to(dtype=dtype)

def _eps_cfg(x_t, t, emb_u, emb_c, cfg_w):
    ud = unet.dtype
    x_t   = x_t.to(dtype=ud)
    emb_u = emb_u.to(dtype=ud)
    emb_c = emb_c.to(dtype=ud)
    eps_u = unet(x_t, t, encoder_hidden_states=emb_u).sample
    eps_c = unet(x_t, t, encoder_hidden_states=emb_c).sample
    return eps_u + float(cfg_w) * (eps_c - eps_u)

def _loss_terms(z_next_hat, x_target):
    loss_full = torch.mean((z_next_hat - x_target) ** 2)
    loss_out  = torch.mean(((z_next_hat - x_target) * mask_out_hard) ** 2)  # preserve background
    loss_in   = torch.mean(((z_next_hat - x_target) * mask_in_hard ) ** 2)  # tiny reg in object
    return loss_full, loss_out, loss_in

def _loss_total(loss_full, loss_out, loss_in):
    if LOSS_MODE == "full":
        return loss_full
    elif LOSS_MODE == "outside+tiny-inside":
        return loss_out + float(LAMBDA_IN) * loss_in
    else:
        raise ValueError(f"Unknown LOSS_MODE={LOSS_MODE}")

uncond_opt = []
z_path     = []

emb_uncond_base = emb_uncond_init
emb_cond_base   = emb_cond

print("=== Cell 8 (NTI, FREE path) start ===")
_tinfo("emb_uncond_base", emb_uncond_base)
_tinfo("inv_xt[0] (xT)", inv_xt[0])
_tinfo("inv_xt[-1] (x0)", inv_xt[-1])
_tinfo("mask_out_hard", mask_out_hard)
_tinfo("mask_in_hard", mask_in_hard)
print(f"CFG_W_TRAIN={CFG_W_TRAIN} | NTI_ITERS={NTI_ITERS} NTI_LR={NTI_LR} | USE_FP32_UNET_OPT={USE_FP32_UNET_OPT}")
print(f"LOSS_MODE={LOSS_MODE} | LAMBDA_IN={LAMBDA_IN} | EPS_STOP={EPS_STOP} | SKIP_IF_GOOD={SKIP_IF_GOOD}")

z = inv_xt[0].to(device=device, dtype=DTYPE).clone()
z_path.append(z.detach().clone())

u_prev_fp32 = None
u_init_fp32 = emb_uncond_base.detach().clone().to(device=device, dtype=torch.float32)

skipped_steps = 0
total_inner_iters = 0

if USE_FP32_UNET_OPT:
    _set_unet_dtype(torch.float32)

for k, t in enumerate(tqdm(ts, desc="NTI null-opt (FREE path)")):
    if k == (NUM_STEPS - 1):
        u_last = uncond_opt[-1].detach().clone() if len(uncond_opt) > 0 else emb_uncond_base.detach().clone().to(device=device, dtype=DTYPE)
        uncond_opt.append(u_last)
        z_path.append(z.detach().clone())
        print(f"[NTI] k={k:02d} t={int(t):4d} (last) -> store u_last, stop.")
        _tinfo(f"u[{k}]", u_last)
        break

    x_target = inv_xt[k+1].to(device=device, dtype=DTYPE)

    u0 = uncond_opt[-1].detach().clone().to(device=device, dtype=torch.float32) if len(uncond_opt) > 0 else u_init_fp32.detach().clone()

    if SKIP_IF_GOOD:
        with torch.no_grad():
            u0_cast = u0.to(dtype=unet.dtype if USE_FP32_UNET_OPT else DTYPE)
            eps0 = _eps_cfg(z, t, u0_cast, emb_cond_base, CFG_W_TRAIN)
            z_next0 = pipe.scheduler.step(eps0.to(DTYPE), t, z).prev_sample
            loss_full0, loss_out0, loss_in0 = _loss_terms(z_next0, x_target)
            thr = _skip_thr_for_k(k, NUM_STEPS)
            if float(loss_out0) < float(thr):
                skipped_steps += 1
                u_final = u0_cast.detach().to(dtype=DTYPE)
                uncond_opt.append(u_final)

                eps = _eps_cfg(z, t, u_final, emb_cond_base, CFG_W_TRAIN)
                z = pipe.scheduler.step(eps.to(DTYPE), t, z).prev_sample
                z_path.append(z.detach().clone())

                if k < 3 or k in [10, 20, 30, 40]:
                    print(f"[NTI-SKIP] k={k:02d} t={int(t):4d} out={float(loss_out0):.3e} thr={thr:.1e} "
                          f"full={float(loss_full0):.3e} in={float(loss_in0):.3e}")
                continue

    u = u0
    u.requires_grad_(True)
    opt = torch.optim.Adam([u], lr=NTI_LR)

    last_full = last_out = last_in = None

    for it in range(int(NTI_ITERS)):
        total_inner_iters += 1
        opt.zero_grad(set_to_none=True)

        u_cast = u.to(dtype=unet.dtype if USE_FP32_UNET_OPT else DTYPE)
        eps = _eps_cfg(z, t, u_cast, emb_cond_base, CFG_W_TRAIN)
        z_next_hat = pipe.scheduler.step(eps.to(DTYPE), t, z).prev_sample

        loss_full, loss_out, loss_in = _loss_terms(z_next_hat, x_target)
        loss = _loss_total(loss_full, loss_out, loss_in)

        if (LAMBDA_U_SMOOTH > 0.0) and (u_prev_fp32 is not None):
            loss = loss + float(LAMBDA_U_SMOOTH) * torch.mean((u - u_prev_fp32) ** 2)

        loss.backward()
        opt.step()

        if NTI_CLAMP is not None:
            with torch.no_grad():
                u.clamp_(-float(NTI_CLAMP), float(NTI_CLAMP))

        last_full, last_out, last_in = loss_full.detach(), loss_out.detach(), loss_in.detach()

        if float(loss.detach()) <= float(EPS_STOP):
            break

    u_final = u.detach().to(device=device, dtype=DTYPE)
    uncond_opt.append(u_final)

    with torch.no_grad():
        eps = _eps_cfg(z, t, u_final, emb_cond_base, CFG_W_TRAIN)
        z = pipe.scheduler.step(eps.to(DTYPE), t, z).prev_sample
    z_path.append(z.detach().clone())

    if k < 3 or k in [10, 20, 30, 40]:
        print(f"[NTI] k={k:02d} t={int(t):4d} "
              f"loss(full/out/in)={float(last_full):.3e}/{float(last_out):.3e}/{float(last_in):.3e} "
              f"loss_total={float(_loss_total(last_full, last_out, last_in)):.3e} "
              f"finite(u/z)={is_finite(u_final)}/{is_finite(z)}")
        _tinfo(f"u[{k}]", u_final)

if USE_FP32_UNET_OPT:
    _set_unet_dtype(_unet_dtype_orig)

print("NTI null-opt done.")
print("  len(uncond_opt) =", len(uncond_opt), " (should be NUM_STEPS)")
print("  len(z_path)     =", len(z_path), " (should be NUM_STEPS+1)")
print("  skipped_steps   =", skipped_steps, "| total_inner_iters =", total_inner_iters)

with torch.no_grad():
    z_free_x0 = z_path[-1].to(device=device, dtype=DTYPE)
    tgt_x0    = inv_xt[-1].to(device=device, dtype=DTYPE)

    mse_full = torch.mean((z_free_x0 - tgt_x0) ** 2).item()
    mse_out  = torch.mean(((z_free_x0 - tgt_x0) * mask_out_hard) ** 2).item()
    mse_in   = torch.mean(((z_free_x0 - tgt_x0) * mask_in_hard ) ** 2).item()

print(f"[FINAL] MSE(z_free_x0, inv_xt[-1]) full/out(bg)/in(obj) = {mse_full:.3e} / {mse_out:.3e} / {mse_in:.3e}")

try:
    with torch.no_grad():
        img_tgt  = vae_decode_fp32(tgt_x0)[0].float().cpu()
        img_free = vae_decode_fp32(z_free_x0)[0].float().cpu()

    def _to_pil_uint8(x_chw):
        x = x_chw.permute(1,2,0).numpy()
        x = np.nan_to_num(x, nan=0.0, posinf=1.0, neginf=0.0)
        return (x * 255.0).clip(0, 255).astype(np.uint8)

    plt.figure(figsize=(12,5))
    plt.subplot(1,2,1); plt.imshow(_to_pil_uint8(img_tgt));  plt.axis("off"); plt.title("Target x0 (inv_xt[-1])")
    plt.subplot(1,2,2); plt.imshow(_to_pil_uint8(img_free)); plt.axis("off"); plt.title("Recon FREE x0")
    plt.tight_layout()
    plt.show()
except Exception as e:
    print("[MONITOR WARN] Could not decode images. Error:", repr(e))


In [ ]:
# ============================================================
# Decoder Self-Attention Masking
# ============================================================

from diffusers.models.attention_processor import Attention
import torch, torch.nn as nn_mod, types

HAM_DEBUG_PRINT_ONCE = False  # set True if you want one-time sim stats

class HAMEditor:
    def __init__(self, mask_tok_8, mask_tok_16, mask_tok_32, mask_tok_64,
                 sam_steps=9, sam_scale=0.3, ham_end_step=10**9):
        self.mask_8  = mask_tok_8
        self.mask_16 = mask_tok_16
        self.mask_32 = mask_tok_32
        self.mask_64 = mask_tok_64
        self.sam_steps = int(sam_steps)
        self.sam_scale = float(sam_scale)
        self.ham_end_step = int(ham_end_step)

        self.cur_step = 0
        self.apply_flags = None    # shape [B] (0/1 per batch item)
        self.strength = 0.0        # 0..1

    def _pick_mask(self, tok_len):
        if tok_len == 8*8:   return self.mask_8
        if tok_len == 16*16: return self.mask_16
        if tok_len == 32*32: return self.mask_32
        if tok_len == 64*64: return self.mask_64
        return None

    def set_step(self, step_idx: int):
        self.cur_step = int(step_idx)

    def set_apply_flags(self, flags_b):
        self.apply_flags = flags_b

    def set_strength(self, s: float):
        self.strength = float(max(0.0, min(1.0, s)))

    # ---------------- Original SAM/HAM mask builder (unchanged logic) ----------------
    def build_masks_and_logits(self, sim_orig, B, H, Q, K):
        if self.apply_flags is None:
            return ("NONE", None, None, None, None)
        if self.cur_step > self.ham_end_step:
            return ("NONE", None, None, None, None)
        if self.strength <= 0.0:
            return ("NONE", None, None, None, None)

        m = self._pick_mask(K)
        if m is None:
            return ("NONE", None, None, None, None)

        if not torch.is_tensor(self.apply_flags):
            flags = torch.tensor(self.apply_flags, device=sim_orig.device, dtype=torch.float32)
        else:
            flags = self.apply_flags.to(sim_orig.device).float()
        flags = flags.view(-1)[:B]
        flags_bh = flags.repeat_interleave(H)

        very_neg = -1e4 if sim_orig.dtype in (torch.float16, torch.bfloat16) else -1e9
        mK = m.to(sim_orig.device).bool().view(1,1,K).expand(B*H, Q, K)
        sim_bg = sim_orig.masked_fill(mK, very_neg)

        if self.cur_step <= self.sam_steps:
            sim_fg = (self.sam_scale * sim_orig).masked_fill(mK, very_neg)
            mQ = m.to(sim_orig.device, dtype=sim_orig.dtype).view(1,Q,1).expand(B*H, Q, 1)
            return ("SAM", sim_bg, sim_fg, mQ, flags_bh)

        return ("HAM", sim_bg, None, None, flags_bh)


def register_decoder_attention_patch(unet, editor: HAMEditor):
    printed = {"done": False}

    def apply_to_out(attn: Attention, x):
        if isinstance(attn.to_out, nn_mod.ModuleList):
            for layer in attn.to_out:
                x = layer(x)
            return x
        return attn.to_out(x)

    def softmax_fp32(sim):
        return torch.softmax(sim.float(), dim=-1).to(sim.dtype)

    def patched_forward(attn: Attention, hidden_states, encoder_hidden_states=None,
                       attention_mask=None, temb=None, **kwargs):
        residual = hidden_states

        hs = hidden_states
        if getattr(attn, "spatial_norm", None) is not None:
            hs = attn.spatial_norm(hs, temb)
        if getattr(attn, "group_norm", None) is not None:
            hs = attn.group_norm(hs)

        q = attn.to_q(hs)

        if encoder_hidden_states is None:
            k_in = hs
        else:
            k_in = encoder_hidden_states
            if getattr(attn, "norm_cross", False) and hasattr(attn, "norm_encoder_hidden_states"):
                k_in = attn.norm_encoder_hidden_states(k_in)

        k = attn.to_k(k_in)
        v = attn.to_v(k_in)

        B, Qlen, C = q.shape
        H = attn.heads
        D = C // H

        def reshape(x):
            return x.view(B, -1, H, D).permute(0,2,1,3).reshape(B*H, -1, D)

        qh = reshape(q); kh = reshape(k); vh = reshape(v)
        sim = torch.bmm(qh, kh.transpose(1,2)) * attn.scale
        Klen = sim.shape[-1]

        if HAM_DEBUG_PRINT_ONCE and (not printed["done"]):
            printed["done"] = True
            s = torch.nan_to_num(sim.float(), nan=0.0, posinf=0.0, neginf=0.0)
            print("[HAM DEBUG] sim stats:",
                  "min", float(s.min()), "max", float(s.max()), "mean", float(s.mean()),
                  "Klen", Klen, "Qlen", Qlen)

        out_no = torch.bmm(softmax_fp32(sim), vh)

        # SAM/HAM affects ONLY decoder self-attn (encoder_hidden_states is None)
        if encoder_hidden_states is None:
            mode, sim_bg, sim_fg, mQ, flags_bh = editor.build_masks_and_logits(sim, B, H, Qlen, Klen)
            if mode != "NONE":
                s = float(editor.strength)

                if mode == "SAM":
                    out_bg = torch.bmm(softmax_fp32(sim_bg), vh)
                    out_fg = torch.bmm(softmax_fp32(sim_fg), vh)
                    out_sam = out_fg * mQ + out_bg * (1.0 - mQ)
                    out_mod = out_sam
                else:
                    out_ham = torch.bmm(softmax_fp32(sim_bg), vh)
                    out_mod = out_ham

                out = out_no
                fb = flags_bh.view(B*H)
                idx = (fb > 0.5).nonzero(as_tuple=True)[0]
                if idx.numel() > 0:
                    out = out.clone()
                    out[idx] = (1.0 - s) * out_no[idx] + s * out_mod[idx]
                    out[idx] = torch.nan_to_num(out[idx])
            else:
                out = out_no
        else:
            out = out_no

        out = out.view(B, H, Qlen, D).permute(0,2,1,3).reshape(B, Qlen, H*D)
        out = apply_to_out(attn, out)

        if getattr(attn, "residual_connection", False):
            out = out + residual
        out = out / getattr(attn, "rescale_output_factor", 1.0)
        return out

    patched = 0
    for n, m in unet.named_modules():
        if isinstance(m, Attention) and ("up_blocks" in n):
            m.forward = types.MethodType(lambda self, *args, **kwargs: patched_forward(self, *args, **kwargs), m)
            patched += 1
    print("Patched decoder Attention modules (up_blocks):", patched)


# SAM_STEPS = 9
editor = HAMEditor(
    mask_tok_8.to(device), mask_tok_16.to(device), mask_tok_32.to(device), mask_tok_64.to(device),
    sam_steps=SAM_STEPS,
    sam_scale=SAM_SCALE,
    ham_end_step=10**9,
)
register_decoder_attention_patch(unet, editor)


In [ ]:
# ============================================================
# Primary Object Removal
# ============================================================


pipe.enable_attention_slicing()

NUM_STEPS = int(NUM_STEPS)
pipe.scheduler.set_timesteps(NUM_STEPS, device=device)
ts = pipe.scheduler.timesteps

def is_finite(x):
    return bool(torch.isfinite(x).all())

def eps_cfg(eps_u, eps_c, w):
    return eps_u + w * (eps_c - eps_u)

def anchor_next(k):
    # anchor for outside locking: use pivotal inversion trajectory
    if k < (len(ts) - 1):
        return inv_xt[k+1].to(device=device, dtype=DTYPE)
    return inv_xt[k].to(device=device, dtype=DTYPE)

def _to_pil(x_chw):
    x = x_chw.permute(1,2,0).numpy()
    x = np.nan_to_num(x, nan=0.0, posinf=1.0, neginf=0.0)
    return Image.fromarray((x*255).clip(0,255).astype(np.uint8))

# Masks (HARD ONLY)
mask_in_hard  = mask_in_hard.to(device=device, dtype=DTYPE).clamp(0,1)
mask_out_hard = mask_out_hard.to(device=device, dtype=DTYPE).clamp(0,1)

# HAM schedule
# HAM_PEAK = 1.0
# HAM_START_FRAC = 0.25
# HAM_RAMP_END_FRAC = 0.80

def attn_strength(k, N):
    k0 = int(HAM_START_FRAC * (N - 1))
    k1 = int(HAM_RAMP_END_FRAC * (N - 1))
    if k <= k0: return 0.0
    if k >= k1: return float(HAM_PEAK)
    x = (k - k0) / max(1, (k1 - k0))
    return float(HAM_PEAK) * float(0.5 - 0.5 * np.cos(np.pi * x))

def run_unet_cfg_singlepass(lat, t, emb_u, emb_c, strength, k_step, apply_on_cond=True):
    editor.set_step(int(k_step))
    editor.set_strength(float(strength))

    if apply_on_cond:
        editor.set_apply_flags(torch.tensor([0.0, 1.0], device=device))
    else:
        editor.set_apply_flags(None)

    latB = lat.repeat(2, 1, 1, 1)
    embB = torch.cat([emb_u, emb_c], dim=0)

    with torch.no_grad():
        eps2 = unet(latB, t, encoder_hidden_states=embB).sample

    eps_u, eps_c = eps2.chunk(2, dim=0)
    return eps_u, eps_c

# CFG_W = 9.5

z = inv_xt[0].clone()

print("\n=== Cell 10 start (EDIT, HARD-lock outside object) ===")
_tinfo("z_init(inv_xt[0])", z)
print(f"CFG_W={CFG_W}")
_tinfo("mask_in_hard(obj)", mask_in_hard)

for k, t in enumerate(tqdm(ts, desc="EDIT(v3 single-pass, HARD outside lock)")):
    emb_u = uncond_opt[k].to(device=device, dtype=DTYPE)

    s_attn = attn_strength(k, NUM_STEPS)
    eps_u, eps_c = run_unet_cfg_singlepass(
        lat=z, t=t, emb_u=emb_u, emb_c=emb_cond,
        strength=s_attn, k_step=k, apply_on_cond=True
    )
    eps = eps_cfg(eps_u, eps_c, CFG_W)

    if not is_finite(eps):
        print(f"[WARN] non-finite eps at k={k} -> forcing eps=eps_u")
        eps = eps_u

    z_next = pipe.scheduler.step(eps, t, z).prev_sample

    # ✅ HARD lock outside object
    anc = anchor_next(k)
    z_next = z_next * mask_in_hard + anc * mask_out_hard

    if k < 3 or k % 10 == 0 or k == (NUM_STEPS - 1):
        with torch.no_grad():
            diff_out = (z_next - anc) * mask_out_hard
            outside_mse = torch.mean(diff_out**2).item()
            outside_max = torch.max(torch.abs(diff_out)).item()
            inside_delta = torch.mean(torch.abs((z_next - z) * mask_in_hard)).item()
        print(f"[DBG] k={k:02d} t={int(t):4d} s_attn={s_attn:.3f} "
              f"outside_mse={outside_mse:.3e} outside_max={outside_max:.3e} inside_mean_abs_delta(obj)={inside_delta:.3e}")
        _tinfo("  z_next", z_next)

    if not is_finite(z_next):
        print(f"[WARN] non-finite z at k={k} -> reset (inside prev, outside anchor)")
        z_next = z * mask_in_hard + anc * mask_out_hard

    z = z_next

    if (k % 10) == 0 and device == "cuda":
        torch.cuda.empty_cache()

with torch.no_grad():
    out_pre = vae_decode(z)[0].float().cpu()

plt.figure(figsize=(6,6))
plt.imshow(_to_pil(out_pre)); plt.axis("off"); plt.title("Edited (pre-refine)")
plt.show()


In [ ]:
# ============================================================
# Localized RENOISE–DENOISE Refinement
# ============================================================



# REFINE_ROUNDS = 10
# TREF_FRAC     = 0.5
# CFG_REF       = 9.5

# SEED_REFINE = 0
torch.manual_seed(SEED_REFINE)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED_REFINE)

mask_in_hard  = mask_in_hard.to(device=device, dtype=DTYPE).clamp(0,1)
mask_out_hard = mask_out_hard.to(device=device, dtype=DTYPE).clamp(0,1)

z_refine = z.detach().clone()

for r in range(REFINE_ROUNDS):
    tref_k = int(TREF_FRAC * (NUM_STEPS - 1))
    tref_k = max(0, min(tref_k, NUM_STEPS - 2))
    tref = ts[tref_k]

    print(f"\n[REFINE ROUND {r+1}/{REFINE_ROUNDS}] tref_k={tref_k} timestep={int(tref)} CFG_REF={CFG_REF}")

    noise = torch.randn_like(z_refine)
    z_noisy_full = pipe.scheduler.add_noise(z_refine, noise, tref)

    # ✅ Hard-lock outside at refine start to inversion trajectory at tref_k
    anc0 = inv_xt[tref_k].to(device=device, dtype=DTYPE)
    z_cur = z_noisy_full * mask_in_hard + anc0 * mask_out_hard

    for k in tqdm(range(tref_k, NUM_STEPS), desc=f"REFINE(v3 single-pass) r{r+1}"):
        t = ts[k]
        emb_u = uncond_opt[k].to(device=device, dtype=DTYPE)
        s_attn = attn_strength(k, NUM_STEPS)

        eps_u, eps_c = run_unet_cfg_singlepass(
            lat=z_cur, t=t, emb_u=emb_u, emb_c=emb_cond,
            strength=s_attn, k_step=k, apply_on_cond=True
        )
        eps = eps_cfg(eps_u, eps_c, CFG_REF)

        if not is_finite(eps):
            print(f"[WARN] non-finite eps in refine at k={k} -> fallback eps_u")
            eps = eps_u

        z_next = pipe.scheduler.step(eps, t, z_cur).prev_sample

        # ✅ Hard-lock outside object
        anc = anchor_next(k)
        z_next = z_next * mask_in_hard + anc * mask_out_hard

        if k in [tref_k, tref_k+1] or (k % 10 == 0) or (k == NUM_STEPS-1):
            with torch.no_grad():
                diff_out = (z_next - anc) * mask_out_hard
                outside_mse = torch.mean(diff_out**2).item()
                outside_max = torch.max(torch.abs(diff_out)).item()
            print(f"[REF DBG] k={k:02d} t={int(t):4d} s_attn={s_attn:.3f} "
                  f"outside_mse={outside_mse:.3e} outside_max={outside_max:.3e}")

        z_cur = z_next

    z_refine = z_cur

with torch.no_grad():
    out_ref = vae_decode(z_refine)[0].float().cpu()

plt.figure(figsize=(12,6))
plt.subplot(1,2,1)
plt.imshow(_to_pil(out_pre)); plt.axis("off")
plt.title(f"Pre-refine | CFG_W={CFG_W}")

plt.subplot(1,2,2)
plt.imshow(_to_pil(out_ref)); plt.axis("off")
plt.title(f"Refined | REFINE_ROUNDS={REFINE_ROUNDS} | TREF_FRAC={TREF_FRAC} | CFG_REF={CFG_REF}")

plt.show()


In [ ]:
# ============================================================
# Evaluation NTI Refined
# ============================================================


# ----------------------------
# 0) Inputs sanity
# ----------------------------
assert "img_pil_512" in globals(), "img_pil_512 missing"
assert "mask_dil" in globals(), "mask_dil missing"
assert "out_ref" in globals(), "out_ref missing (Cell 11 output)"

mask_np = (mask_dil > 0).astype(np.uint8)
H, W = mask_np.shape
assert (H, W) == (512, 512), f"mask_dil must be 512x512, got {(H,W)}"

device_eval = "cuda" if torch.cuda.is_available() else "cpu"

def _pil_to_t01_chw(pil_img):
    arr = np.array(pil_img).astype(np.float32) / 255.0  # HWC
    t = torch.from_numpy(arr).permute(2,0,1)            # CHW
    return t.clamp(0,1)

x_in  = _pil_to_t01_chw(img_pil_512).to(device_eval)        # [3,512,512]
x_ref = out_ref.to(device_eval).clamp(0,1)                  # [3,512,512]

m = torch.from_numpy(mask_np[None,None].astype(np.float32)).to(device_eval)  # [1,1,H,W]
m = m.clamp(0,1)
m01 = m[0]  # [1,H,W]

# ----------------------------
# 1) Boundary band builder
# ----------------------------
def make_boundary_band(mask_01_b1hw, band_px=12):
    k = int(max(1, band_px))
    dil = F.max_pool2d(mask_01_b1hw, kernel_size=2*k+1, stride=1, padding=k)
    ero = 1.0 - F.max_pool2d(1.0 - mask_01_b1hw, kernel_size=2*k+1, stride=1, padding=k)
    band = (dil - ero).clamp(0,1)
    inner = (band * mask_01_b1hw).clamp(0,1)
    outer = (band * (1.0 - mask_01_b1hw)).clamp(0,1)
    return band, inner, outer

# ----------------------------
# 2) Edge/Gradient seam metrics
# ----------------------------
def sobel_grad_mag(x_chw):
    g = (0.2989*x_chw[0] + 0.5870*x_chw[1] + 0.1140*x_chw[2]).unsqueeze(0).unsqueeze(0)
    kx = torch.tensor([[-1,0,1],[-2,0,2],[-1,0,1]], dtype=torch.float32, device=g.device).view(1,1,3,3)
    ky = torch.tensor([[-1,-2,-1],[0,0,0],[1,2,1]], dtype=torch.float32, device=g.device).view(1,1,3,3)
    gx = F.conv2d(g, kx, padding=1)
    gy = F.conv2d(g, ky, padding=1)
    return torch.sqrt(gx*gx + gy*gy + 1e-12)

def boundary_seam_score(x_chw, mask_b1hw, band_px=12):
    band, inner, outer = make_boundary_band(mask_b1hw, band_px=band_px)
    grad = sobel_grad_mag(x_chw)
    eps = 1e-8
    gi = (grad * inner).sum() / (inner.sum() + eps)
    go = (grad * outer).sum() / (outer.sum() + eps)
    mismatch_abs = torch.abs(gi - go)
    mismatch_rel = mismatch_abs / (go + eps)
    return float(gi), float(go), float(mismatch_abs), float(mismatch_rel)

def band_color_jump(x_chw, mask_b1hw, band_px=12):
    band, inner, outer = make_boundary_band(mask_b1hw, band_px=band_px)
    eps = 1e-8
    xi = (x_chw.unsqueeze(0) * inner.repeat(1,3,1,1)).sum(dim=(0,2,3)) / (inner.sum() + eps)
    xo = (x_chw.unsqueeze(0) * outer.repeat(1,3,1,1)).sum(dim=(0,2,3)) / (outer.sum() + eps)
    return float(torch.sqrt(((xi - xo)**2).sum() + 1e-12))

# ----------------------------
# 3) Neighborhood consistency (outside rings)
# ----------------------------
def ring_consistency(x_chw, mask_b1hw, inner_px=0, ring_px=20):
    k0 = int(max(0, inner_px))
    k1 = int(max(1, inner_px + ring_px))
    k2 = int(max(1, inner_px + 2*ring_px))

    dil0 = F.max_pool2d(mask_b1hw, kernel_size=2*k0+1, stride=1, padding=k0) if k0 > 0 else mask_b1hw
    dil1 = F.max_pool2d(mask_b1hw, kernel_size=2*k1+1, stride=1, padding=k1)
    dil2 = F.max_pool2d(mask_b1hw, kernel_size=2*k2+1, stride=1, padding=k2)

    near = (dil1 - dil0).clamp(0,1) * (1.0 - mask_b1hw)
    far  = (dil2 - dil1).clamp(0,1) * (1.0 - mask_b1hw)

    eps = 1e-8
    xn = (x_chw.unsqueeze(0) * near.repeat(1,3,1,1)).sum(dim=(0,2,3)) / (near.sum() + eps)
    xf = (x_chw.unsqueeze(0) * far.repeat(1,3,1,1)).sum(dim=(0,2,3)) / (far.sum() + eps)
    rgb_diff = torch.sqrt(((xn - xf)**2).sum() + 1e-12)

    grad = sobel_grad_mag(x_chw)
    gn = (grad * near).sum() / (near.sum() + eps)
    gf = (grad * far ).sum() / (far.sum()  + eps)
    grad_diff = torch.abs(gn - gf)
    return float(rgb_diff), float(grad_diff)

# ----------------------------
# 4) Focus images (patch vs context)
# ----------------------------
def _make_focus_images(x_chw, mask_01_hw, gray=0.5):
    patch_focus = x_chw * mask_01_hw + gray * (1.0 - mask_01_hw)
    ctx_focus   = x_chw * (1.0 - mask_01_hw) + gray * mask_01_hw
    return patch_focus.clamp(0,1), ctx_focus.clamp(0,1)

def _tchw_to_pil(x_chw):
    x = (x_chw.detach().clamp(0,1).permute(1,2,0).cpu().numpy() * 255.0).astype(np.uint8)
    from PIL import Image
    return Image.fromarray(x)

# ----------------------------
# 5) CLIP background alignment (robust) + fallback ResNet18
# ----------------------------
def clip_bg_alignment(patch_chw, ctx_chw, device="cpu"):
    try:
        from transformers import CLIPProcessor, CLIPModel
        model_id = "openai/clip-vit-base-patch32"
        proc = CLIPProcessor.from_pretrained(model_id)
        mdl  = CLIPModel.from_pretrained(model_id).to(device).eval()

        pil_patch = _tchw_to_pil(patch_chw)
        pil_ctx   = _tchw_to_pil(ctx_chw)

        with torch.no_grad():
            inps = proc(images=[pil_patch, pil_ctx], return_tensors="pt").to(device)
            pv = inps["pixel_values"]
            vout = mdl.vision_model(pixel_values=pv)
            pooled = vout.pooler_output
            feats  = mdl.visual_projection(pooled)
            feats  = feats / (feats.norm(dim=-1, keepdim=True) + 1e-12)
            sim = (feats[0] * feats[1]).sum()
        return float(sim), "CLIP"
    except Exception as e:
        print("[EVAL WARN] CLIP failed -> fallback to ResNet18. Error:", repr(e))
        try:
            import torchvision.models as models
            res = models.resnet18(weights=models.ResNet18_Weights.DEFAULT).to(device).eval()
            feat_extractor = torch.nn.Sequential(*list(res.children())[:-1]).to(device).eval()

            mean = torch.tensor([0.485,0.456,0.406], device=device).view(3,1,1)
            std  = torch.tensor([0.229,0.224,0.225], device=device).view(3,1,1)

            def _prep(x_chw):
                return ((x_chw.clamp(0,1) - mean)/std).unsqueeze(0)

            with torch.no_grad():
                f1 = feat_extractor(_prep(patch_chw)).view(-1)
                f2 = feat_extractor(_prep(ctx_chw)).view(-1)
                f1 = f1 / (f1.norm() + 1e-12)
                f2 = f2 / (f2.norm() + 1e-12)
                sim = (f1 * f2).sum()
            return float(sim), "ResNet18"
        except Exception as e2:
            print("[EVAL WARN] ResNet18 fallback failed:", repr(e2))
            return float("nan"), "None"

# ----------------------------
# 6) Local feature distance (ResNet50)
# ----------------------------
def local_feature_distance(patch_chw, ctx_chw, device="cpu"):
    try:
        import torchvision.models as models
        res = models.resnet50(weights=models.ResNet50_Weights.DEFAULT).to(device).eval()
        feat_extractor = torch.nn.Sequential(*list(res.children())[:-1]).to(device).eval()

        mean = torch.tensor([0.485,0.456,0.406], device=device).view(3,1,1)
        std  = torch.tensor([0.229,0.224,0.225], device=device).view(3,1,1)

        def _prep(x_chw):
            return ((x_chw.clamp(0,1) - mean)/std).unsqueeze(0)

        with torch.no_grad():
            f1 = feat_extractor(_prep(patch_chw)).view(-1)
            f2 = feat_extractor(_prep(ctx_chw)).view(-1)
            f1n = f1 / (f1.norm() + 1e-12)
            f2n = f2 / (f2.norm() + 1e-12)
            d = torch.sqrt(((f1n - f2n)**2).sum() + 1e-12)
        return float(d), "ResNet50"
    except Exception as e:
        print("[EVAL WARN] ResNet50 feature distance failed:", repr(e))
        return float("nan"), "None"

# ----------------------------
# 7) Ring-mix seam metrics: SSIM_ring + LPIPS_ring
# ----------------------------
def _to_gray_np_rgb01(x_chw_cpu):
    x = x_chw_cpu.clamp(0,1)
    y = (0.2989*x[0] + 0.5870*x[1] + 0.1140*x[2]).detach().cpu().numpy().astype(np.float32)
    return y

def _ssim_gray_np(a, b):
    import cv2
    C1 = (0.01**2); C2 = (0.03**2)
    mu_a = cv2.GaussianBlur(a, (11,11), 1.5)
    mu_b = cv2.GaussianBlur(b, (11,11), 1.5)
    mu_a2 = mu_a*mu_a; mu_b2 = mu_b*mu_b; mu_ab = mu_a*mu_b
    sigma_a2 = cv2.GaussianBlur(a*a, (11,11), 1.5) - mu_a2
    sigma_b2 = cv2.GaussianBlur(b*b, (11,11), 1.5) - mu_b2
    sigma_ab = cv2.GaussianBlur(a*b, (11,11), 1.5) - mu_ab
    ssim_map = ((2*mu_ab + C1) * (2*sigma_ab + C2)) / ((mu_a2 + mu_b2 + C1) * (sigma_a2 + sigma_b2 + C2) + 1e-12)
    return float(ssim_map.mean())

def build_ring_mix(x_ref_chw, x_in_chw, mask_b1hw, ring_px=8):
    band, _, _ = make_boundary_band(mask_b1hw, band_px=int(max(1, ring_px)))
    band3 = band.repeat(1,3,1,1)[0]  # [3,H,W]
    ring_mix = x_ref_chw * band3 + x_in_chw * (1.0 - band3)
    return ring_mix.clamp(0,1), band

def _get_lpips_net(device):
    global _LPIPS_NET_CACHED
    try:
        _LPIPS_NET_CACHED
    except NameError:
        _LPIPS_NET_CACHED = None

    if _LPIPS_NET_CACHED is not None:
        return _LPIPS_NET_CACHED

    try:
        import lpips
    except Exception:
        try:
            import sys, subprocess
            print("[EVAL INFO] Installing lpips...")
            subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "lpips"])
            import lpips
        except Exception as e2:
            print("[EVAL WARN] LPIPS not available:", repr(e2))
            _LPIPS_NET_CACHED = None
            return None

    net = lpips.LPIPS(net="alex").to(device).eval()
    _LPIPS_NET_CACHED = net
    return net

@torch.no_grad()
def lpips_ring_score(orig_chw, ringmix_chw, device):
    net = _get_lpips_net(device)
    if net is None:
        return float("nan")
    o = (orig_chw.clamp(0,1) * 2 - 1).unsqueeze(0)
    p = (ringmix_chw.clamp(0,1) * 2 - 1).unsqueeze(0)
    return float(net(o, p).mean().item())

# ============================================================
# RUN EVAL (REFINED ONLY)
# ============================================================
# BAND_PX = 12
# RING_PX = 20
# RING_SEAM_PX = 8

gi, go, g_abs, g_rel = boundary_seam_score(x_ref, m, band_px=BAND_PX)
c_jump = band_color_jump(x_ref, m, band_px=BAND_PX)
rgb_ring, grad_ring = ring_consistency(x_ref, m, inner_px=0, ring_px=RING_PX)

patch_focus, ctx_focus = _make_focus_images(x_ref, m01)
sim_bg, sim_kind = clip_bg_alignment(patch_focus, ctx_focus, device=device_eval)
d_feat, feat_kind = local_feature_distance(patch_focus, ctx_focus, device=device_eval)

ring_mix, _ = build_ring_mix(x_ref, x_in, m, ring_px=RING_SEAM_PX)
ssim_ring = _ssim_gray_np(_to_gray_np_rgb01(x_in.detach().cpu()),
                          _to_gray_np_rgb01(ring_mix.detach().cpu()))
lpips_ring = lpips_ring_score(x_in, ring_mix, device=device_eval)

# ----------------------------
# Pretty metric block (string)
# ----------------------------
metric_lines = [
    "FINAL OUTPUT EVAL (REFINED)",
    f"mask: area={float(mask_np.mean()):.4f} | band_px={BAND_PX} | ring_px={RING_PX} | ring_seam_px={RING_SEAM_PX}",
    "",
    f"Boundary seam (grad): in={gi:.4f} out={go:.4f}",
    f"  abs_mismatch={g_abs:.4f} | rel_mismatch={g_rel:.4f}  (lower better)",
    f"Boundary seam (color jump L2): {c_jump:.4f}            (lower better)",
    f"Neighborhood consistency (outside rings): rgb_l2={rgb_ring:.4f} | grad_abs={grad_ring:.4f} (lower better)",
    f"BG alignment (patch vs context): sim={sim_bg:.4f} ({sim_kind}) (higher better)",
    f"Local feature distance (patch vs context): d={d_feat:.4f} ({feat_kind}) (lower better)",
    f"SSIM_ring (orig vs ring_mix): {ssim_ring:.4f}          (higher better)",
    f"LPIPS_ring (orig vs ring_mix): {lpips_ring:.4f}        (lower better)",
]
metrics_text = "\n".join(metric_lines)

# ----------------------------
# Show: final refined image + metrics next to it
# ----------------------------
plt.figure(figsize=(16, 6))

# Left: final refined image
plt.subplot(1, 2, 1)
plt.imshow(_tchw_to_pil(x_ref))
plt.axis("off")
plt.title("Final refined output (out_ref)")

# Right: metrics panel
plt.subplot(1, 2, 2)
plt.axis("off")
plt.title("Evaluation metrics")
plt.text(
    0.0, 1.0, metrics_text,
    va="top", ha="left",
    family="monospace",
    fontsize=11,
)

plt.tight_layout()
plt.show()

# Still print to console (so it's in logs too)
print("\n==================== FINAL OUTPUT EVAL (REFINED) ====================")
print(metrics_text)
print("====================================================================\n")


In [ ]:
# ============================================================
# No-NTI Ablation
# ============================================================



pipe.enable_attention_slicing()

# ----------------------------
# Timesteps / alignment
# ----------------------------
NUM_STEPS = int(NUM_STEPS)
pipe.scheduler.set_timesteps(NUM_STEPS, device=device)
ts = pipe.scheduler.timesteps  # descending; inv_xt[0] most noisy

# ----------------------------
# Helpers
# ----------------------------
def is_finite(x):
    return bool(torch.isfinite(x).all())

def eps_cfg(eps_u, eps_c, w):
    return eps_u + w * (eps_c - eps_u)

def anchor_next(k):
    return inv_xt[k+1].to(device=device, dtype=DTYPE) if (k < len(ts)-1) else inv_xt[k].to(device=device, dtype=DTYPE)

def _to_pil(x_chw):
    x = x_chw.permute(1,2,0).detach().cpu().numpy()
    x = np.nan_to_num(x, nan=0.0, posinf=1.0, neginf=0.0)
    return Image.fromarray((x * 255).clip(0,255).astype(np.uint8))

# ----------------------------
# Masks (HARD ONLY)
# ----------------------------
mask_in_hard  = mask_in_hard.to(device=device, dtype=DTYPE).clamp(0,1)
mask_out_hard = mask_out_hard.to(device=device, dtype=DTYPE).clamp(0,1)

# ----------------------------
# SAM/HAM strength schedule
# ----------------------------
# HAM_PEAK = 1.0
# HAM_START_FRAC = 0.25
# HAM_RAMP_END_FRAC = 0.80

def attn_strength(k, N):
    k0 = int(HAM_START_FRAC * (N - 1))
    k1 = int(HAM_RAMP_END_FRAC * (N - 1))
    if k <= k0:
        return 0.0
    if k >= k1:
        return float(HAM_PEAK)
    x = (k - k0) / max(1, (k1 - k0))
    return float(HAM_PEAK) * float(0.5 - 0.5 * np.cos(np.pi * x))

# ----------------------------
# v3 single-pass CFG call
# ----------------------------
def run_unet_cfg_singlepass(lat, t, emb_u, emb_c, strength, k_step, apply_on_cond=True):
    """
    One UNet call for CFG using batch=[uncond, cond].
    SAM/HAM patch affects ONLY decoder self-attn and only when apply_flags is set and strength>0.
    """
    editor.set_step(int(k_step))
    editor.set_strength(float(strength))

    if apply_on_cond:
        editor.set_apply_flags(torch.tensor([0.0, 1.0], device=device))
    else:
        editor.set_apply_flags(None)

    latB = lat.repeat(2, 1, 1, 1)
    embB = torch.cat([emb_u, emb_c], dim=0)

    with torch.no_grad():
        eps2 = unet(latB, t, encoder_hidden_states=embB).sample

    eps_u, eps_c = eps2.chunk(2, dim=0)
    return eps_u, eps_c

# ----------------------------
# Main knobs
# ----------------------------
# CFG_W = 9.5

print("\n=== Cell 10 (NO-NTI) start — trajectory-anchored, HARD outside lock ===")
print(f"CFG_W={CFG_W}")
_tinfo("emb_uncond_init (fixed)", emb_uncond_init)
_tinfo("mask_in_hard", mask_in_hard)

# Start at the same xT from inversion
z = inv_xt[0].clone()

for k, t in enumerate(tqdm(ts, desc="EDIT(NO-NTI, strict-outside)")):

    emb_u = emb_uncond_init.to(device=device, dtype=DTYPE)

    # SAM/HAM on cond branch
    s_attn = attn_strength(k, NUM_STEPS)
    eps_u, eps_c = run_unet_cfg_singlepass(
        lat=z, t=t, emb_u=emb_u, emb_c=emb_cond,
        strength=s_attn, k_step=k, apply_on_cond=True
    )
    eps = eps_cfg(eps_u, eps_c, CFG_W)

    if not is_finite(eps):
        print(f"[WARN] non-finite eps at k={k} -> forcing eps=eps_u")
        eps = eps_u

    z_next = pipe.scheduler.step(eps, t, z).prev_sample

    # ✅ strict outside lock to inversion trajectory (HARD)
    anc = anchor_next(k)
    z_next = z_next * mask_in_hard + anc * mask_out_hard

    if k < 3 or k % 10 == 0 or k == (NUM_STEPS - 1):
        with torch.no_grad():
            diff_out = (z_next - anc) * mask_out_hard
            outside_mse = torch.mean(diff_out**2).item()
            inside_delta = torch.mean(torch.abs((z_next - z) * mask_in_hard)).item()
        print(f"[DBG] k={k:02d} t={int(t):4d} s_attn={s_attn:.3f} outside_mse={outside_mse:.3e} inside_mean_abs_delta={inside_delta:.3e}")

    if not is_finite(z_next):
        print(f"[WARN] non-finite z at k={k} -> hard reset (inside prev, outside anchor)")
        z_next = z * mask_in_hard + anc * mask_out_hard

    z = z_next

    if (k % 10) == 0 and device == "cuda":
        torch.cuda.empty_cache()

with torch.no_grad():
    out_pre_noNTI = vae_decode(z)[0].float().cpu()

plt.figure(figsize=(6,6))
plt.imshow(_to_pil(out_pre_noNTI)); plt.axis("off")
plt.title("Edited (pre-refine) — NO NTI (trajectory-anchored)")
plt.show()


In [ ]:
# ============================================================
# NO-NTI — STRONG REFINE
# ============================================================


# REFINE_ROUNDS = 10
# TREF_FRAC     = 0.5
# CFG_REF       = 9.5

# SEED_REFINE = 0
torch.manual_seed(SEED_REFINE)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED_REFINE)

# Masks (HARD ONLY)
mask_in_hard  = mask_in_hard.to(device=device, dtype=DTYPE).clamp(0,1)
mask_out_hard = mask_out_hard.to(device=device, dtype=DTYPE).clamp(0,1)

z_refine = z.detach().clone()  # start from NO-NTI edited latent

for r in range(REFINE_ROUNDS):
    tref_k = int(TREF_FRAC * (NUM_STEPS - 1))
    tref_k = max(0, min(tref_k, NUM_STEPS - 2))
    tref = ts[tref_k]

    print(f"\n[REFINE NO-NTI ROUND {r+1}/{REFINE_ROUNDS}] tref_k={tref_k} timestep={int(tref)} CFG_REF={CFG_REF}")

    noise = torch.randn_like(z_refine)
    z_noisy_full = pipe.scheduler.add_noise(z_refine, noise, tref)

    # ✅ lock outside to inversion trajectory at tref_k (HARD)
    anc0 = inv_xt[tref_k].to(device=device, dtype=DTYPE)
    z_cur = z_noisy_full * mask_in_hard + anc0 * mask_out_hard

    for k in tqdm(range(tref_k, NUM_STEPS), desc=f"REFINE(NO-NTI) r{r+1}"):
        t = ts[k]

        # ✅ NO NTI: fixed unconditional embedding
        emb_u = emb_uncond_init.to(device=device, dtype=DTYPE)

        s_attn = attn_strength(k, NUM_STEPS)

        eps_u, eps_c = run_unet_cfg_singlepass(
            lat=z_cur, t=t, emb_u=emb_u, emb_c=emb_cond,
            strength=s_attn, k_step=k, apply_on_cond=True
        )
        eps = eps_cfg(eps_u, eps_c, CFG_REF)

        if not is_finite(eps):
            print(f"[WARN] non-finite eps in refine at k={k} -> fallback to eps_u")
            eps = eps_u

        z_next = pipe.scheduler.step(eps, t, z_cur).prev_sample

        # ✅ strict outside lock to inversion trajectory (HARD)
        anc = anchor_next(k)
        z_next = z_next * mask_in_hard + anc * mask_out_hard

        if k in [tref_k, tref_k+1] or (k % 10 == 0) or (k == NUM_STEPS-1):
            with torch.no_grad():
                diff_out = (z_next - anc) * mask_out_hard
                outside_mse = torch.mean(diff_out**2).item()
                inside_delta = torch.mean(torch.abs((z_next - z_cur) * mask_in_hard)).item()
            print(f"[REF DBG] k={k:02d} t={int(t):4d} s_attn={s_attn:.3f} outside_mse={outside_mse:.3e} inside_mean_abs_delta={inside_delta:.3e}")

        z_cur = z_next

    z_refine = z_cur

with torch.no_grad():
    out_ref_noNTI = vae_decode(z_refine)[0].float().cpu()

plt.figure(figsize=(12,6))
plt.subplot(1,2,1)
plt.imshow(_to_pil(out_pre_noNTI)); plt.axis("off")
plt.title(f"Pre-refine (NO NTI) | CFG_W={CFG_W}")

plt.subplot(1,2,2)
plt.imshow(_to_pil(out_ref_noNTI)); plt.axis("off")
plt.title(f"Refined (NO NTI) | R={REFINE_ROUNDS} | TREF_FRAC={TREF_FRAC} | CFG_REF={CFG_REF}")

plt.show()


In [ ]:
# ============================================================
# EVAL CELL (NO-NTI)
# Uses ONLY the refined NO-NTI result
# ============================================================


# ----------------------------
# 0) Inputs sanity
# ----------------------------
assert "img_pil_512" in globals(), "img_pil_512 missing"
assert "mask_dil" in globals(), "mask_dil missing"
assert "out_ref_noNTI" in globals(), "out_ref_noNTI missing (NO-NTI Cell 11 output)"

mask_np = (mask_dil > 0).astype(np.uint8)
H, W = mask_np.shape
assert (H, W) == (512, 512), f"mask_dil must be 512x512, got {(H,W)}"

device_eval = "cuda" if torch.cuda.is_available() else "cpu"

def _pil_to_t01_chw(pil_img):
    arr = np.array(pil_img).astype(np.float32) / 255.0  # HWC
    t = torch.from_numpy(arr).permute(2,0,1)            # CHW
    return t.clamp(0,1)

x_in  = _pil_to_t01_chw(img_pil_512).to(device_eval)           # [3,512,512]
x_ref = out_ref_noNTI.to(device_eval).clamp(0,1)               # [3,512,512]

m = torch.from_numpy(mask_np[None,None].astype(np.float32)).to(device_eval)  # [1,1,H,W]
m = m.clamp(0,1)
m01 = m[0]  # [1,H,W]

# ----------------------------
# 1) Boundary band builder
# ----------------------------
def make_boundary_band(mask_01_b1hw, band_px=12):
    k = int(max(1, band_px))
    dil = F.max_pool2d(mask_01_b1hw, kernel_size=2*k+1, stride=1, padding=k)
    ero = 1.0 - F.max_pool2d(1.0 - mask_01_b1hw, kernel_size=2*k+1, stride=1, padding=k)
    band = (dil - ero).clamp(0,1)
    inner = (band * mask_01_b1hw).clamp(0,1)
    outer = (band * (1.0 - mask_01_b1hw)).clamp(0,1)
    return band, inner, outer

# ----------------------------
# 2) Edge/Gradient seam metrics
# ----------------------------
def sobel_grad_mag(x_chw):
    g = (0.2989*x_chw[0] + 0.5870*x_chw[1] + 0.1140*x_chw[2]).unsqueeze(0).unsqueeze(0)
    kx = torch.tensor([[-1,0,1],[-2,0,2],[-1,0,1]], dtype=torch.float32, device=g.device).view(1,1,3,3)
    ky = torch.tensor([[-1,-2,-1],[0,0,0],[1,2,1]], dtype=torch.float32, device=g.device).view(1,1,3,3)
    gx = F.conv2d(g, kx, padding=1)
    gy = F.conv2d(g, ky, padding=1)
    return torch.sqrt(gx*gx + gy*gy + 1e-12)

def boundary_seam_score(x_chw, mask_b1hw, band_px=12):
    band, inner, outer = make_boundary_band(mask_b1hw, band_px=band_px)
    grad = sobel_grad_mag(x_chw)
    eps = 1e-8
    gi = (grad * inner).sum() / (inner.sum() + eps)
    go = (grad * outer).sum() / (outer.sum() + eps)
    mismatch_abs = torch.abs(gi - go)
    mismatch_rel = mismatch_abs / (go + eps)
    return float(gi), float(go), float(mismatch_abs), float(mismatch_rel)

def band_color_jump(x_chw, mask_b1hw, band_px=12):
    band, inner, outer = make_boundary_band(mask_b1hw, band_px=band_px)
    eps = 1e-8
    xi = (x_chw.unsqueeze(0) * inner.repeat(1,3,1,1)).sum(dim=(0,2,3)) / (inner.sum() + eps)
    xo = (x_chw.unsqueeze(0) * outer.repeat(1,3,1,1)).sum(dim=(0,2,3)) / (outer.sum() + eps)
    return float(torch.sqrt(((xi - xo)**2).sum() + 1e-12))

# ----------------------------
# 3) Neighborhood consistency (outside rings)
# ----------------------------
def ring_consistency(x_chw, mask_b1hw, inner_px=0, ring_px=20):
    k0 = int(max(0, inner_px))
    k1 = int(max(1, inner_px + ring_px))
    k2 = int(max(1, inner_px + 2*ring_px))

    dil0 = F.max_pool2d(mask_b1hw, kernel_size=2*k0+1, stride=1, padding=k0) if k0 > 0 else mask_b1hw
    dil1 = F.max_pool2d(mask_b1hw, kernel_size=2*k1+1, stride=1, padding=k1)
    dil2 = F.max_pool2d(mask_b1hw, kernel_size=2*k2+1, stride=1, padding=k2)

    near = (dil1 - dil0).clamp(0,1) * (1.0 - mask_b1hw)
    far  = (dil2 - dil1).clamp(0,1) * (1.0 - mask_b1hw)

    eps = 1e-8
    xn = (x_chw.unsqueeze(0) * near.repeat(1,3,1,1)).sum(dim=(0,2,3)) / (near.sum() + eps)
    xf = (x_chw.unsqueeze(0) * far.repeat(1,3,1,1)).sum(dim=(0,2,3)) / (far.sum() + eps)
    rgb_diff = torch.sqrt(((xn - xf)**2).sum() + 1e-12)

    grad = sobel_grad_mag(x_chw)
    gn = (grad * near).sum() / (near.sum() + eps)
    gf = (grad * far ).sum() / (far.sum()  + eps)
    grad_diff = torch.abs(gn - gf)
    return float(rgb_diff), float(grad_diff)

# ----------------------------
# 4) Focus images (patch vs context)
# ----------------------------
def _make_focus_images(x_chw, mask_01_hw, gray=0.5):
    patch_focus = x_chw * mask_01_hw + gray * (1.0 - mask_01_hw)
    ctx_focus   = x_chw * (1.0 - mask_01_hw) + gray * mask_01_hw
    return patch_focus.clamp(0,1), ctx_focus.clamp(0,1)

def _tchw_to_pil(x_chw):
    x = (x_chw.detach().clamp(0,1).permute(1,2,0).cpu().numpy() * 255.0).astype(np.uint8)
    from PIL import Image
    return Image.fromarray(x)

# ----------------------------
# 5) CLIP background alignment (robust) + fallback ResNet18
# ----------------------------
def clip_bg_alignment(patch_chw, ctx_chw, device="cpu"):
    try:
        from transformers import CLIPProcessor, CLIPModel
        model_id = "openai/clip-vit-base-patch32"
        proc = CLIPProcessor.from_pretrained(model_id)
        mdl  = CLIPModel.from_pretrained(model_id).to(device).eval()

        pil_patch = _tchw_to_pil(patch_chw)
        pil_ctx   = _tchw_to_pil(ctx_chw)

        with torch.no_grad():
            inps = proc(images=[pil_patch, pil_ctx], return_tensors="pt").to(device)
            pv = inps["pixel_values"]
            vout = mdl.vision_model(pixel_values=pv)
            pooled = vout.pooler_output
            feats  = mdl.visual_projection(pooled)
            feats  = feats / (feats.norm(dim=-1, keepdim=True) + 1e-12)
            sim = (feats[0] * feats[1]).sum()
        return float(sim), "CLIP"
    except Exception as e:
        print("[EVAL WARN] CLIP failed -> fallback to ResNet18. Error:", repr(e))
        try:
            import torchvision.models as models
            res = models.resnet18(weights=models.ResNet18_Weights.DEFAULT).to(device).eval()
            feat_extractor = torch.nn.Sequential(*list(res.children())[:-1]).to(device).eval()

            mean = torch.tensor([0.485,0.456,0.406], device=device).view(3,1,1)
            std  = torch.tensor([0.229,0.224,0.225], device=device).view(3,1,1)

            def _prep(x_chw):
                return ((x_chw.clamp(0,1) - mean)/std).unsqueeze(0)

            with torch.no_grad():
                f1 = feat_extractor(_prep(patch_chw)).view(-1)
                f2 = feat_extractor(_prep(ctx_chw)).view(-1)
                f1 = f1 / (f1.norm() + 1e-12)
                f2 = f2 / (f2.norm() + 1e-12)
                sim = (f1 * f2).sum()
            return float(sim), "ResNet18"
        except Exception as e2:
            print("[EVAL WARN] ResNet18 fallback failed:", repr(e2))
            return float("nan"), "None"

# ----------------------------
# 6) Local feature distance (ResNet50)
# ----------------------------
def local_feature_distance(patch_chw, ctx_chw, device="cpu"):
    try:
        import torchvision.models as models
        res = models.resnet50(weights=models.ResNet50_Weights.DEFAULT).to(device).eval()
        feat_extractor = torch.nn.Sequential(*list(res.children())[:-1]).to(device).eval()

        mean = torch.tensor([0.485,0.456,0.406], device=device).view(3,1,1)
        std  = torch.tensor([0.229,0.224,0.225], device=device).view(3,1,1)

        def _prep(x_chw):
            return ((x_chw.clamp(0,1) - mean)/std).unsqueeze(0)

        with torch.no_grad():
            f1 = feat_extractor(_prep(patch_chw)).view(-1)
            f2 = feat_extractor(_prep(ctx_chw)).view(-1)
            f1n = f1 / (f1.norm() + 1e-12)
            f2n = f2 / (f2.norm() + 1e-12)
            d = torch.sqrt(((f1n - f2n)**2).sum() + 1e-12)
        return float(d), "ResNet50"
    except Exception as e:
        print("[EVAL WARN] ResNet50 feature distance failed:", repr(e))
        return float("nan"), "None"

# ----------------------------
# 7) Ring-mix seam metrics: SSIM_ring + LPIPS_ring
# ----------------------------
def _to_gray_np_rgb01(x_chw_cpu):
    x = x_chw_cpu.clamp(0,1)
    y = (0.2989*x[0] + 0.5870*x[1] + 0.1140*x[2]).detach().cpu().numpy().astype(np.float32)
    return y

def _ssim_gray_np(a, b):
    import cv2
    C1 = (0.01**2); C2 = (0.03**2)
    mu_a = cv2.GaussianBlur(a, (11,11), 1.5)
    mu_b = cv2.GaussianBlur(b, (11,11), 1.5)
    mu_a2 = mu_a*mu_a; mu_b2 = mu_b*mu_b; mu_ab = mu_a*mu_b
    sigma_a2 = cv2.GaussianBlur(a*a, (11,11), 1.5) - mu_a2
    sigma_b2 = cv2.GaussianBlur(b*b, (11,11), 1.5) - mu_b2
    sigma_ab = cv2.GaussianBlur(a*b, (11,11), 1.5) - mu_ab
    ssim_map = ((2*mu_ab + C1) * (2*sigma_ab + C2)) / ((mu_a2 + mu_b2 + C1) * (sigma_a2 + sigma_b2 + C2) + 1e-12)
    return float(ssim_map.mean())

def build_ring_mix(x_ref_chw, x_in_chw, mask_b1hw, ring_px=8):
    band, _, _ = make_boundary_band(mask_b1hw, band_px=int(max(1, ring_px)))
    band3 = band.repeat(1,3,1,1)[0]
    ring_mix = x_ref_chw * band3 + x_in_chw * (1.0 - band3)
    return ring_mix.clamp(0,1), band

def _get_lpips_net(device):
    global _LPIPS_NET_CACHED
    try:
        _LPIPS_NET_CACHED
    except NameError:
        _LPIPS_NET_CACHED = None

    if _LPIPS_NET_CACHED is not None:
        return _LPIPS_NET_CACHED

    try:
        import lpips
    except Exception:
        try:
            import sys, subprocess
            print("[EVAL INFO] Installing lpips...")
            subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "lpips"])
            import lpips
        except Exception as e2:
            print("[EVAL WARN] LPIPS not available:", repr(e2))
            _LPIPS_NET_CACHED = None
            return None

    net = lpips.LPIPS(net="alex").to(device).eval()
    _LPIPS_NET_CACHED = net
    return net

@torch.no_grad()
def lpips_ring_score(orig_chw, ringmix_chw, device):
    net = _get_lpips_net(device)
    if net is None:
        return float("nan")
    o = (orig_chw.clamp(0,1) * 2 - 1).unsqueeze(0)
    p = (ringmix_chw.clamp(0,1) * 2 - 1).unsqueeze(0)
    return float(net(o, p).mean().item())

# ============================================================
# RUN EVAL (REFINED NO-NTI ONLY)
# ============================================================
# BAND_PX = 12
# RING_PX = 20
# RING_SEAM_PX = 8

gi, go, g_abs, g_rel = boundary_seam_score(x_ref, m, band_px=BAND_PX)
c_jump = band_color_jump(x_ref, m, band_px=BAND_PX)
rgb_ring, grad_ring = ring_consistency(x_ref, m, inner_px=0, ring_px=RING_PX)

patch_focus, ctx_focus = _make_focus_images(x_ref, m01)
sim_bg, sim_kind = clip_bg_alignment(patch_focus, ctx_focus, device=device_eval)
d_feat, feat_kind = local_feature_distance(patch_focus, ctx_focus, device=device_eval)

ring_mix, _ = build_ring_mix(x_ref, x_in, m, ring_px=RING_SEAM_PX)
ssim_ring = _ssim_gray_np(_to_gray_np_rgb01(x_in.detach().cpu()),
                          _to_gray_np_rgb01(ring_mix.detach().cpu()))
lpips_ring = lpips_ring_score(x_in, ring_mix, device=device_eval)

metric_lines = [
    "FINAL OUTPUT EVAL (REFINED NO-NTI)",
    f"mask: area={float(mask_np.mean()):.4f} | band_px={BAND_PX} | ring_px={RING_PX} | ring_seam_px={RING_SEAM_PX}",
    "",
    f"Boundary seam (grad): in={gi:.4f} out={go:.4f}",
    f"  abs_mismatch={g_abs:.4f} | rel_mismatch={g_rel:.4f}  (lower better)",
    f"Boundary seam (color jump L2): {c_jump:.4f}            (lower better)",
    f"Neighborhood consistency (outside rings): rgb_l2={rgb_ring:.4f} | grad_abs={grad_ring:.4f} (lower better)",
    f"BG alignment (patch vs context): sim={sim_bg:.4f} ({sim_kind}) (higher better)",
    f"Local feature distance (patch vs context): d={d_feat:.4f} ({feat_kind}) (lower better)",
    f"SSIM_ring (orig vs ring_mix): {ssim_ring:.4f}          (higher better)",
    f"LPIPS_ring (orig vs ring_mix): {lpips_ring:.4f}        (lower better)",
]
metrics_text = "\n".join(metric_lines)

plt.figure(figsize=(16, 6))

plt.subplot(1, 2, 1)
plt.imshow(_tchw_to_pil(x_ref))
plt.axis("off")
plt.title("Final refined output (NO-NTI) — out_ref_noNTI")

plt.subplot(1, 2, 2)
plt.axis("off")
plt.title("Evaluation metrics (NO-NTI)")
plt.text(
    0.0, 1.0, metrics_text,
    va="top", ha="left",
    family="monospace",
    fontsize=11,
)

plt.tight_layout()
plt.show()

print("\n==================== FINAL OUTPUT EVAL (REFINED NO-NTI) ====================")
print(metrics_text)
print("============================================================================\n")

# Optional visualization (helps interpret seam metrics)
try:
    plt.figure(figsize=(14,4))
    plt.subplot(1,3,1); plt.imshow(_tchw_to_pil(patch_focus)); plt.axis("off"); plt.title("Patch-focus (outside gray)")
    plt.subplot(1,3,2); plt.imshow(_tchw_to_pil(ctx_focus));   plt.axis("off"); plt.title("Context-focus (patch gray)")
    plt.subplot(1,3,3); plt.imshow(_tchw_to_pil(ring_mix));    plt.axis("off"); plt.title("Ring-mix (only boundary from output)")
    plt.tight_layout(); plt.show()
except Exception as e:
    print("[EVAL VIS WARN]", repr(e))
